In [2]:
# Importing the NumPy module and giving it the conventional shorthand name np.
import numpy as np 

## Problem 1: Getting started with arrays (2 points)

Implement the two functions `checkerboard` and `transpose` below, see instructions in the corresponding docstrings. 

**IMPORTANT**: DO NOT USE THE BUILT-IN METHODS arr.transpose()  OR arr.T OR THE FUNCTION np.transpose(arr). CREATE IT YOURSELF FROM SCRATCH!

In [53]:
def checkerboard(n):
    """
    Returns a two dimensional n by n checkerboard array C
    that alternates between 0's and 1's, starting with a 0
    in the top-left corner. 
    
    Example
    ----------
    checkerboard(4) =
        [[0 1 0 1]
         [1 0 1 0]
         [0 1 0 1]
         [1 0 1 0]]
    
    Parameters
    ----------
        n (int): Order of the n by n array
        
    Returns
    -------
        C (ndarray): Array filled with a checkerboard pattern
    
    """
    
    # YOUR CODE HERE
    C = np.zeros((n, n), dtype=int)
    for i in range(n):
        if i % 2 == 0:
            value = 0
        else:
            value = 1

        for j in range(n):
            C[i, j] = value
            if value == 0:
                value = 1
            else:
                value = 0
    return C
    
    
def transpose(arr):
    """
    Returns the transpose of a two dimensional array arr 
    
    Parameters
    ----------
        arr (ndarray): Any arbitrary two-dimensional array
        
    Returns
    -------
        arr_T (ndarray): Two-dimensional array arr transposed

    """
    
    # YOUR CODE HERE
    rows = len(arr)
    cols = len(arr[0])
    arr_T = np.zeros((cols, rows), dtype=arr.dtype)
    for i in range(rows):
        for j in range(cols):
            arr_T[j, i] = arr[i, j]
    return arr_T

In [55]:
# Test case
C = checkerboard(5)
print(C, "\n")

arr = np.array([[1, 2, 3], 
                [4, 5, 6], 
                [7, 8, 9]])
print(transpose(arr), "\n")

[[0 1 0 1 0]
 [1 0 1 0 1]
 [0 1 0 1 0]
 [1 0 1 0 1]
 [0 1 0 1 0]] 

[[1 4 7]
 [2 5 8]
 [3 6 9]] 



Expected output:

    [[0 1 0 1 0]
     [1 0 1 0 1]
     [0 1 0 1 0]
     [1 0 1 0 1]
     [0 1 0 1 0]]
     
    [[1 4 7]
     [2 5 8]
     [3 6 9]] 

In [56]:
# AUTOGRADING for checkerboard
C = [[0, 1, 0, 1, 0],
     [1, 0, 1, 0, 1],
     [0, 1, 0, 1, 0],
     [1, 0, 1, 0, 1],
     [0, 1, 0, 1, 0]] 
assert np.allclose(C, checkerboard(5))


In [57]:
# AUTOGRADING for transpose
arr = arr = np.array([[1, 2, 3], 
                      [4, 5, 6], 
                      [7, 8, 9]])
T = arr.transpose()
assert np.allclose(T, transpose(arr))

arr2D = np.arange(1,25).reshape(4,6)
T = arr2D.transpose()
assert np.allclose(T, transpose(arr2D))

import inspect
sig = inspect.signature(transpose)       # get the signature from the function 
source = inspect.getsource(transpose)    # get the source from the function (as a string! Wooow!)

assert not "np.transpose" in source
assert not "arr.transpose" in source
assert not "arr.T" in source

## Standard Gram-Schmidt Process

Recall that every rectangular $n \times m$ ($n \geq m$) matrix $A$ with linearly indepedent columns can be represented as a product
$$A = QR, \qquad \text{or in matrix form} \qquad \begin{bmatrix}\mid & \mid & & \mid \\ v_1 & v_2 & \dots & v_m \\ \mid & \mid & & \mid \end{bmatrix} = \begin{bmatrix}\mid & \mid & & \mid \\ u_1 & u_2 & \dots & u_m \\ \mid & \mid & & \mid \end{bmatrix} \begin{bmatrix} r_{11} & r_{12} & \dots & r_{1m} \\ & r_{22} & \dots & r_{2m}\\ & & \ddots & \vdots \\ & & & r_{mm} \end{bmatrix}$$
where $Q$ is an orthogonal $n \times m$ matrix (i.e. its columns are orthonormal) and $R$ is an upper-triangular $m \times m$ matrix. In this assignment your task will be to implement several methods to achieve this factorization for a given matrix $A$. 

Our first goal is to orthogonalize a system of linearly independent vectors $v_1, \dots, v_m$ (i.e. the columns of some matrix $A$). The standard procedure for this task is the Gram-Schmidt process as covered in the lecture:
\begin{align*}
u_1 &= \frac{v_1}{r_{11}} \\
v_2^\perp &= v_2 - r_{12} u_1, \quad u_2 = \frac{v_2^\perp}{r_{22}} \\
v_3^\perp &= v_3 - r_{13} u_1 - r_{23} u_2, \quad u_3 = \frac{v_2^\perp}{r_{33}} \\
\vdots
\end{align*}
where
\begin{align*}
r_{jj} &= \|v_j - \sum_{i=1}^{j-1} r_{ij} u_i\| &&\quad \text{for }\ 1 \leq j \leq m\ \\
r_{ij} &= \langle v_j, u_i \rangle &&\quad \text{for } 1 \leq i < j \leq m
\end{align*}

## Problem 2: (Classical) Gram-Schmidt Orthogonalization (3 points)

Implement the described Gram-Schmidt algorithm as a function `gram_schmidt_qr` below that takes a rectangular array $A$ and outputs $Q$ and $R$. 

**IMPORTANT**: Do not use any built-in QR functions like `np.linalg.qr` or `scipy.lingalg.qr`. These will give different results anyway as these use a different numerical scheme. 

**HINT**: The following functions for vectors might be useful: `np.linalg.norm` to compute the norm of a 1-D array and `np.inner` to compute the regular dot product between two 1-D arrays.

In [58]:
def gram_schmidt_qr(A):
    """
    Computes the QR factorization of a matrix A with linearly independent columns
    by means of the classical Gram-Schmidt process, where Q is an orthogonal matrix 
    and R is upper-triangular with positive entries on its diagonal. 
    
    Parameters
    ----------
        A (ndarray): A two-dimensional array of size n by m with n >= m
            whose columns are linearly independent
    
    Returns
    -------
        Q (ndarray): A two-dimensional array of size n by m
        R (ndarray): A two-dimensional array of size m by m
        
    """
    
    n, m = A.shape
    Q = np.zeros((n,m))
    R = np.zeros((m,m))
    
    # YOUR CODE HERE
    for j in range(m):
        v = A[:, j].copy()

        for i in range(j):
            R[i, j] = np.inner(A[:, j], Q[:, i])
            v = v - R[i, j] * Q[:, i]

        R[j, j] = np.linalg.norm(v)
        Q[:, j] = v / R[j, j]
    
    return Q, R

In [60]:
# Test case I
np.set_printoptions(suppress=True) # Prints nicer numbers, you may ignore this

A = np.array([[2,2], [1,7], [-2, -8]])
Q, R = gram_schmidt_qr(A)
print(Q, "\n")
print(R)

[[ 0.66666667 -0.66666667]
 [ 0.33333333  0.66666667]
 [-0.66666667 -0.33333333]] 

[[3. 9.]
 [0. 6.]]


Expected output:

    [[ 0.66666667 -0.66666667]
     [ 0.33333333  0.66666667]
     [-0.66666667 -0.33333333]]
     
    [[3. 9.]
     [0. 6.]]

In [61]:
# Test case II
B = np.array([[-1, 4, -1], [1, -1, -1], [1, -1, 1], [-1, 4, -3]])
Q, R = gram_schmidt_qr(B)
print(Q, "\n")
print(R)

[[-0.5  0.5  0.5]
 [ 0.5  0.5 -0.5]
 [ 0.5  0.5  0.5]
 [-0.5  0.5 -0.5]] 

[[ 2. -5.  2.]
 [ 0.  3. -2.]
 [ 0.  0.  2.]]


Expected output:

    [[-0.5  0.5  0.5]
     [ 0.5  0.5 -0.5]
     [ 0.5  0.5  0.5]
     [-0.5  0.5 -0.5]]
     
    [[ 2. -5.  2.]
     [ 0.  3. -2.]
     [ 0.  0.  2.]]

In [62]:
# AUTOGRADER
A = np.array([[2,2], [1,7], [-2, -8]])
Q, R = gram_schmidt_qr(A)
Q_exact, R_exact = 1/3*np.array([[2, -2], [1, 2], [-2, -1]]), np.array([[3, 9], [0, 6]])

assert np.allclose(Q, Q_exact)
assert np.allclose(R, R_exact)

B = np.array([[-1, 4, -1], [1, -1, -1], [1, -1, 1], [-1, 4, -3]])
Q, R = gram_schmidt_qr(B)
Q_exact, R_exact = 1/2 * np.array([[-1, 1, 1], [1, 1, -1], [1, 1, 1], [-1, 1, -1]]), np.array([[2, -5, 2], [0, 3, -2], [0, 0, 2]])

assert np.allclose(Q, Q_exact)
assert np.allclose(R, R_exact)


## Householder Triangularization

Another principal method for computing QR factorizations is Householder triangularization, which is numerically more stable than Gram-Schmidt orthogonalization. The Householder algorithm is a process of "orthogonal triangularization", making a matrix triangular by a sequence of orthogonal matrix operations. 

In contrast to the Gram-Schmidt iteration which applies a succession of elementary triangular matrices $R_i$ on the right of $A$ resulting in an orthogonal matrix, the Householder method applies a succession of elementary orthogonal matrices $Q_i$ on the left of $A$ so that the resulting matrix
$$Q_m \dots Q_2 Q_1 A = R$$
is upper-triangular. The product $Q = Q_1^T Q_2^T \dots Q_m^T$ is also orthogonal, and therefore $A = QR$ is our $QR$ factorization of $A$. We can summarize this method as orthogonal triangularization. 

At the heart of the Householder method are orthogonal matrices $Q_i$ to introduce zeroes as indicated below. In these matrices, the symbol $\times$ represents an entry that is not necesssarily zero, and boldfacing them indicates an entry that has just been changed. Blank entries are zero. 

$$\begin{bmatrix}
\times & \times & \times \\ 
\times & \times & \times \\ 
\times & \times & \times \\ 
\times & \times & \times \\ 
\times & \times & \times 
\end{bmatrix} \xrightarrow{Q_1} \begin{bmatrix}
\pmb{\times} & \pmb{\times} & \pmb{\times} \\ 
\pmb{0} & \pmb{\times} & \pmb{\times} \\ 
\pmb{0} & \pmb{\times} & \pmb{\times} \\ 
\pmb{0} & \pmb{\times} & \pmb{\times} \\ 
\pmb{0} & \pmb{\times} & \pmb{\times} 
\end{bmatrix} \xrightarrow{Q_2} \begin{bmatrix}
\times & \times & \times \\ 
 & \pmb{\times} & \pmb{\times} \\ 
 & \pmb{0} & \pmb{\times} \\ 
 & \pmb{0} & \pmb{\times} \\ 
 & \pmb{0} & \pmb{\times} 
\end{bmatrix} \xrightarrow{Q_3} \begin{bmatrix}
\times & \times & \times \\ 
 & \times & \times \\ 
 &  & \pmb{\times} \\ 
 &  & \pmb{0} \\ 
 &  & \pmb{0} 
\end{bmatrix}$$

In general $Q_k$ operates on rows $k, \dots, n$ and columns $k, \dots, m$. At the beginning of step $k$, there is a block of zeroes in the first $k-1$ columns of these rows. The application of $Q_k$ forms linear combinations of these rows, and the linear combinations of the zero entries remain zero. 

### Householder Reflectors

So how do we construct orthogonal matrices $Q_k$ to introduce zeroes as indicated above? The standard approach is as follows. Each $Q_k$ is chosen to be an orthogonal matrix of the form
$$Q_k = \begin{bmatrix} I & 0 \\ 0 & F \end{bmatrix} \label{eq_Q}\tag{1},$$
where $I$ is the $(k-1) \times (k-1)$ identity matrix and $F$ is an $(n-k+1) \times (n - k + 1)$ orthogonal matrix. The identity matrix will make sure that we don't alter the first $k-1$ columns that already have zeroes introduced in the right replaces, and multiplication by $F$ must introduce zeroes into the $k$th column. The Householder algorithm chooses $F$ to be a particular matrix called a *Householder reflector*. 

Suppose, at the beginning of step $k$, the entries $k, \dots, n$ of the $k$th column are given by a vector $x \in \mathbb{R}^{n-k+1}$. To introduce the correct zeroes into the $k$th column, the Householder reflector $F$ should effect the  following map (recall an orthogonal matrix is length preserving):
$$x = \begin{bmatrix} \times \\ \times \\ \vdots \\ \times \end{bmatrix} \xrightarrow{F} Fx = \begin{bmatrix} \|x\| \\ 0 \\ \vdots \\ 0 \end{bmatrix} = \|x\| e_1.$$
(We shall modify this idea by a $\pm$ sign in a moment). The idea for accomplishing this is indicated in the figure below. 

<img src="https://i.ibb.co/gRhhyzW/householder.png" alt="householder" border="0" width=500></a>

The reflector $F$ will reflect the space $\mathbb{R}^{n-k+1}$ across the hyperplane $H$ orthogonal to the vector $v = \|x\|e_1 - x$. The formula for this reflection can be derived as follows. We know that the vector 
$$Py = \left(I - \frac{v v^T}{v^T v}\right)y = y - v \left(\frac{v^T y}{v^T v}\right) = y - \frac{v}{\|v\|} \left\langle \frac{v}{\|v\|}, y \right\rangle = y - \langle u, y \rangle u$$
is the orthogonal projection of $y$ onto the space $H$. To reflect $y$ across $H$, we must not stop at this point; we must go exactly twice as far in the same direction. The reflection $Fy$ should therefore be
$$Fy = \left(I - 2\frac{v v^T}{v^T v}\right)y = y - 2v \left(\frac{v^T y}{v^T v}\right).$$
Hence the matrix $F$ is 
$$F = I - 2 \frac{v v^T}{v^T v}, \quad \text{with} \quad v = \|x\| e_1 - x.$$

### The Better of Two Reflectors

In the above figure we have simplified matters, for in fact, there are two Householder reflections that will introduce the zeroes needed. In the figure below two alternatives are represented by reflections across two different hyperplanes, $H^+$ and $H^-$. 

<img src="https://i.ibb.co/dDNSbS0/householder2.png" alt="householder2" border="0" width=500></a>

Mathematically either choice of sign is satisfactory. However, numerical stability (insensitivity to rounding errors) dictates that one choice should be taken rather than the other. For numerical stability, it is desirable to reflect $x$ to the vector $z \|x\| e_1$ that is not too close to $x$ itself. To achieve this, we can choose $z = -\text{sign}(x_1)$, where $x_1$ denotes the first component of $x$, so that the reflection vector becomes $v = -\text{sign}(x_1) \|x\| e_1 - x$, or upon clearing the factors $-1$,
$$v = \text{sign}(x_1) \|x\| e_1 + x.$$
To make a complete prescription, we may arbitrarily impose the convention that $\text{sign}(x_1) = 1$ if $x_1 = 0$. 

To see why the choice of sign makes a difference for stability, suppose that the angle between $H^+$ and the $e_1$-axis is very small. Then the vector $v = \|x\|e_1 - x$ is much smaller than $x$ or $\|x\|e_1$. Thus the calculation of $v$ represents a subtraction of nearby quantities and will tend to suffer from cancellation errors. By picking the opposite sign, we avoid such effects by ensuring that $\|v\|$ is never smaller than $\|x\|$. 

**Note**: In this QR factorization the rectangular $n \times m$ ($n \geq m$) matrix $A$ with linearly indepedent columns will be represented as a product
$$A = QR, \qquad \text{or in matrix form} \qquad \begin{bmatrix}\mid & \mid & & \mid \\ v_1 & v_2 & \dots & v_m \\ \mid & \mid & & \mid \end{bmatrix} = \begin{bmatrix}\mid & \mid & & \mid \\ u_1 & u_2 & \dots & u_n \\ \mid & \mid & & \mid \end{bmatrix} \begin{bmatrix} r_{11} & r_{12} & \dots & r_{1m} \\ & r_{22} & \dots & r_{2m}\\ & & \ddots & \vdots \\ & & & r_{mm} \\ & & & \end{bmatrix}$$
where $Q$ is an orthogonal $n \times n$ matrix and $R$ is an upper-triangular $n \times m$ matrix. So when $A$ is not square, $R$ will contain extra rows of zeroes at the bottom. Moreover, we also allow for negative entries along the diagonal (as opposed to the QR factorization we obtain by applying the Gram-Schmidt process). 

## Problem 3: Householder QR Factorization (5 = 2 + 3 points)

Implement both the functions `householder_reflector` and `householder_qr` below. 

**Hint**: The function [`np.outer`](https://numpy.org/doc/stable/reference/generated/numpy.outer.html) might be useful to compute $v v^T$ when we represent $v$ as a 1 dimensional array. The function [`np.linalg.norm`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html) is useful to compute the norm of a vector. The function [`np.inner`](https://numpy.org/doc/stable/reference/generated/numpy.inner.html) is useful to compute the dot product of two vectors. 

You can multiply two 2 dimensional arrays (i.e. matrices) $A$ and $B$ by `A @ B` or by `np.matmul(A,B)`. You can compute the inverse of a matrix by [`np.lingalg.inv`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.inv.html).

In [63]:
def sign(x):
    """
    To be used in householder_qr below in the computation of
        v = sign(x) * norm(x) * e_1 + x
    """
    if(x >= 0):
        return(1)
    else:
        return(-1)

def householder_reflector(v, k, n):
    """
    Returns the orthogonal n by n Householder reflector Q_k as described by equation (1) above,
    corresponding to the hyperplane determined by the vector v.
    
    Parameters
    ----------
        v (ndarray): A 1 dimensional array of length n-k+1. 
        k (int): Denotes the required Householder reflector
                 BEWARE THAT PYTHON INDEXING STARTS AT ZERO,
                 THIS MIGHT BE CONFUSING WITH THE WAY Q_K IS
                 NUMBERED.
        n (int): Number of rows and columns of Q_k
    
    Returns
    -------
        Q_k (ndarray): A 2 dimensional n by n array. 
    
    """
    
    # YOUR CODE HERE
    F = np.eye(len(v)) - 2 * np.outer(v, v) / np.inner(v, v)
    Q_k = np.eye(n)
    Q_k[k-1:, k-1:] = F
    return Q_k

    
def householder_qr(A):
    """
    Computes a QR factorization of a matrix A by means of Householder
    reflectors, where Q is an orthogonal matrix and R is upper-triangular. 
    
    Parameters
    ----------
        A (ndarray): A two-dimensional array of size n by m with n >= m
            whose columns are linearly independent
    
    Returns
    -------
        Q (ndarray): A two-dimensional array of size n by n
        R (ndarray): A two-dimensional array of size n by m
    
    """
    
    # YOUR CODE HERE
    n, m = A.shape
    R = A.astype(float).copy()
    Q = np.eye(n)

    for k in range(1, m + 1):
        x = R[k-1:, k-1]

        e1 = np.zeros(len(x))
        e1[0] = 1

        v = sign(x[0]) * np.linalg.norm(x) * e1 + x

        Q_k = householder_reflector(v, k, n)
        R = Q_k @ R
        Q = Q @ Q_k.T

    return Q, R

In [65]:
# You can use this code cell to play around with your function to make sure
# it does what it is intended to do, i.e. to debug your code. 


In [66]:
# Test case for householder_reflector
v1 = np.array([5, 1, -2])
Q_1 = householder_reflector(v1, 1, 3)
print(Q_1, "\n")

v2 = np.array([10.8, -3.6])
Q_2 = householder_reflector(v2, 2, 3)
print(Q_2)

[[-0.66666667 -0.33333333  0.66666667]
 [-0.33333333  0.93333333  0.13333333]
 [ 0.66666667  0.13333333  0.73333333]] 

[[ 1.   0.   0. ]
 [ 0.  -0.8  0.6]
 [ 0.   0.6  0.8]]


Expected output:

    [[-0.66666667 -0.33333333  0.66666667]
     [-0.33333333  0.93333333  0.13333333]
     [ 0.66666667  0.13333333  0.73333333]]
     
    [[ 1.   0.   0. ]
     [ 0.  -0.8  0.6]
     [ 0.   0.6  0.8]]

In [67]:
# Test case for householder_qr
np.set_printoptions(suppress=True) # Prints nicer numbers, you may ignore this

A = np.array([[2,2], [1,7], [-2, -8]])
Q, R = householder_qr(A)
print(Q, "\n")
print(R)

[[-0.66666667  0.66666667  0.33333333]
 [-0.33333333 -0.66666667  0.66666667]
 [ 0.66666667  0.33333333  0.66666667]] 

[[-3. -9.]
 [-0. -6.]
 [-0.  0.]]


Expected output:

    [[-0.66666667  0.66666667  0.33333333]
     [-0.33333333 -0.66666667  0.66666667]
     [ 0.66666667  0.33333333  0.66666667]]
     
    [[-3. -9.]
     [-0. -6.]
     [-0.  0.]]

In [68]:
# Test case II
B = np.array([[-1, 4, -1], [1, -1, -1], [1, -1, 1], [-1, 4, -3]])
Q, R = householder_qr(B)
print(Q, "\n")
print(R)

[[-0.5 -0.5 -0.5 -0.5]
 [ 0.5 -0.5  0.5 -0.5]
 [ 0.5 -0.5 -0.5  0.5]
 [-0.5 -0.5  0.5  0.5]] 

[[ 2. -5.  2.]
 [-0. -3.  2.]
 [-0. -0. -2.]
 [-0.  0.  0.]]


Expected output:

    [[-0.5 -0.5 -0.5 -0.5]
     [ 0.5 -0.5  0.5 -0.5]
     [ 0.5 -0.5 -0.5  0.5]
     [-0.5 -0.5  0.5  0.5]]
     
    [[ 2. -5.  2.]
     [ 0. -3.  2.]
     [ 0.  0. -2.]
     [ 0. -0.  0.]]

In [69]:
# AUTOGRADER for householder_reflector
v1 = np.array([5, 1, -2])
Q_1 = householder_reflector(v1, 1, 3)
assert np.allclose(Q_1, 1/15 * np.array([[-10, -5, 10], [-5, 14, 2], [10, 2, 11]]))

v2 = np.array([10.8, -3.6])
Q_2 = householder_reflector(v2, 2, 3)
assert np.allclose(Q_2, 1/5 * np.array([[5, 0, 0], [0, -4, 3], [0, 3, 4]]))

In [70]:
# AUTOGRADER for householder_qr
A = np.array([[2,2], [1,7], [-2, -8]])
Q, R = householder_qr(A)
Q_exact, R_exact = 1/3*np.array([[-2, 2, 1], [-1, -2, 2], [2, 1, 2]]), np.array([[-3, -9], [0, -6], [0, 0]])

assert np.allclose(Q, Q_exact)
assert np.allclose(R, R_exact)

B = np.array([[-1, 4, -1], [1, -1, -1], [1, -1, 1], [-1, 4, -3]])
Q, R = householder_qr(B)

Q_exact = 1/2 * np.array([[-1, -1, -1, -1], [1, -1, 1, -1], [1, -1, -1, 1], [-1, -1, 1, 1]]) 
R_exact = np.array([[2, -5, 2], [0, -3, 2], [0, 0, -2], [0, 0, 0]])

assert np.allclose(Q, Q_exact)
assert np.allclose(R, R_exact)
